# Week 1 · Dashboard data using tradinglab
هذا النوتبوك الآن يستخدم نفس بيئة `tradinglab` التي تعتمد عليها باقي الأسبوع الأول.
نؤدي هنا نفس استراتيجية الشراء والبيع الأسبوعية الخاصة بك، لكن نقرأ البيانات من `DataFeed` الموجود في المشروع.

In [ ]:
import sys, os
from pathlib import Path

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed, load_egx30_returns
from tradinglab.charting import plot_equity, plot_portfolio_composition_discrete, turnover_summary
from tradinglab.report import report
from tradinglab import metrics

plt.style.use('ggplot')
print('startup ok')

## 1. Data and universe
هذا هو نفس الاستراتيجية الأسبوعية الخاصة بك، لكن الآن نستخدم `DataFeed` من `tradinglab` على كل الأسهم في `data/egx`.\n
نأخذ سعر الإغلاق الأسبوعي ونحسب العوائد الأسبوعية قبل أن ننفذ إشارات الشراء والبيع.

In [ ]:
DATA_PATH = Path('data/egx')

BUY_DROP = -0.05
SELL_GAIN = 0.10
BUY_AMOUNT = 5.0
SELL_AMOUNT = 10.0
INITIAL_CASH = 10000.0

feed = DataFeed.from_dir(DATA_PATH)
prices = pd.DataFrame(feed.close, index=feed.dates, columns=feed.symbols)
weekly_prices = prices.resample('W-FRI').last().ffill()
weekly_returns = weekly_prices.pct_change()

print('universe symbols:', len(feed.symbols))
print(feed.symbols)
print('weekly history rows:', len(weekly_prices))

In [ ]:
cash = INITIAL_CASH
positions = pd.Series(0.0, index=weekly_prices.columns)
transactions = []
portfolio_history = []
weights_history = []
buy_count = []
sell_count = []

for idx, date in enumerate(weekly_prices.index):
    week_price = weekly_prices.loc[date]
    week_return = weekly_returns.loc[date] if idx > 0 else pd.Series(np.zeros(len(weekly_prices.columns)), index=weekly_prices.columns)

    if idx == 0:
        portfolio_history.append({'Date': date, 'Cash': cash, 'Holdings': 0.0, 'Portfolio': cash})
        weights_history.append(np.zeros(feed.n_assets))
        buy_count.append(0)
        sell_count.append(0)
        continue

    buys = 0
    sells = 0

    for symbol in feed.symbols:
        rtn = week_return[symbol]
        if pd.isna(rtn):
            continue
        if rtn <= BUY_DROP:
            qty = BUY_AMOUNT / week_price[symbol]
            positions[symbol] += qty
            cash -= BUY_AMOUNT
            buys += 1
            transactions.append({
                'Date': date,
                'Symbol': symbol,
                'Action': 'BUY',
                'Price': week_price[symbol],
                'Quantity': qty,
                'Amount': BUY_AMOUNT,
            })

    for symbol in feed.symbols:
        rtn = week_return[symbol]
        if pd.isna(rtn):
            continue
        if rtn >= SELL_GAIN:
            qty = SELL_AMOUNT / week_price[symbol]
            qty = min(qty, positions[symbol])
            if qty > 0:
                positions[symbol] -= qty
                cash += qty * week_price[symbol]
                sells += 1
                transactions.append({
                    'Date': date,
                    'Symbol': symbol,
                    'Action': 'SELL',
                    'Price': week_price[symbol],
                    'Quantity': qty,
                    'Amount': qty * week_price[symbol],
                })

    holdings_value = (positions * week_price).sum()
    portfolio_value = cash + holdings_value
    weights = (positions * week_price) / portfolio_value if portfolio_value > 0 else np.zeros(feed.n_assets)

    portfolio_history.append({'Date': date, 'Cash': cash, 'Holdings': holdings_value, 'Portfolio': portfolio_value})
    weights_history.append(weights.to_numpy())
    buy_count.append(buys)
    sell_count.append(sells)

portfolio = pd.DataFrame(portfolio_history).set_index('Date')
weights = np.vstack(weights_history)
transactions = pd.DataFrame(transactions)

## 2. Run the backtest
هذه هي نفس فكرة backtest في notebook 4، لكن المحرك هنا هو استراتيجية الشراء والبيع الأسبوعية الأصلية.

In [ ]:
benchmark_returns = weekly_returns.reindex(portfolio.index).mean(axis=1).fillna(0.0).to_numpy()
benchmark = np.cumprod(1 + benchmark_returns)
portfolio_curve = portfolio['Portfolio'].to_numpy() / INITIAL_CASH
portfolio_returns = portfolio['Portfolio'].pct_change().fillna(0.0).to_numpy()

result = {
    'dates': portfolio.index,
    'portfolio': portfolio_curve,
    'benchmark': benchmark,
    'weights': weights,
    'portfolio_returns': portfolio_returns,
}

print('final portfolio value:', f"{portfolio['Portfolio'].iloc[-1]:,.2f} EGP")
print('total return:', f'{portfolio_curve[-1] - 1:+.1%}')
print('total buys:', sum(buy_count), '| total sells:', sum(sell_count))
print('total trades:', len(transactions))
print('average stocks held:', f"{turnover_summary(weights)['avg_stocks_held']:.2f}")

## 4. Equity curve
هذا الرسم يظهر أداء الاستراتيجية الأسبوعية مقابل مؤشر الوزن المتساوي في نفس النافذة الزمنية.

In [ ]:
fig, axes = plt.subplots(len(feed.symbols), 1, figsize=(11, 2.2 * len(feed.symbols)), sharex=True)
for ax, symbol in zip(axes, feed.symbols):
    ax.plot(weekly_prices.index, weekly_prices[symbol], linewidth=1.0)
    ax.set_ylabel(symbol, rotation=0, labelpad=20)
    ax.grid(alpha=0.3)
fig.suptitle('Your universe — weekly closing prices')
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

plot_portfolio_composition_discrete(result['weights'], result['dates'], feed.symbols, title='Portfolio weights over time')
plt.show()

In [ ]:
START_CAPITAL = INITIAL_CASH
curve = result['portfolio'] * START_CAPITAL
peak = np.maximum.accumulate(curve)
drawdown_series = (peak - curve) / peak

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(result['dates'], curve, label='strategy')
ax1.plot(result['dates'], peak, '--', color='gray', label='running peak')
ax1.set_ylabel('EGP')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_title('Your weekly strategy equity curve vs its running peak')

ax2.fill_between(result['dates'], -drawdown_series, 0, color='red', alpha=0.5)
ax2.set_title('Drawdown relative to the running peak')
ax2.set_ylabel('drawdown')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

worst_dd = metrics.max_drawdown(result['portfolio_returns'])
worst_dd_egp = peak.max() * worst_dd
print(f'worst drawdown: {worst_dd:.1%} (~{worst_dd_egp:,.0f} EGP)')

In [ ]:
report(result, title='Weekly buy/sell strategy vs equal-weight benchmark', start_capital=INITIAL_CASH)

In [ ]:
egx30_returns = load_egx30_returns('data/egx30.csv', result['dates'])
if egx30_returns is not None:
    egx30_curve = np.cumprod(1 + egx30_returns) * INITIAL_CASH
    plt.figure(figsize=(11, 5))
    plt.plot(result['dates'], result['portfolio'] * INITIAL_CASH, label='strategy')
    plt.plot(result['dates'], result['benchmark'] * INITIAL_CASH, label='equal-weight benchmark', linestyle='--')
    plt.plot(result['dates'], egx30_curve, label='real EGX30 index', linestyle=':')
    plt.title('Weekly strategy vs equal-weight benchmark vs real EGX30')
    plt.ylabel('EGP')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.gcf().autofmt_xdate()
    plt.show()
else:
    print('EGX30 benchmark data not available for this date range.')